# IOAI — 2025 Residential Camp Ml Hdb Storey (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/teodora-dimitrova/Singapore-Team-Selection"
CLONE = "Singapore-Team-Selection"
SUBDIR = "IOAI 2025 Residential camp task/ ML_ResidentialCamp2025"
WORKDIR = "IOAI 2025 Residential camp task/ ML_ResidentialCamp2025"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# HDB Resale — Storey-Range Prediction

> **Singapore IOAI 2025 — Residential Camp (ML)**

Predict the **`storey_range`** (17 ordered bands, e.g. `01 TO 03` … `49 TO 51`) of Singapore
HDB resale flats from listing features. Multiclass classification; score = **accuracy**.

Files: `IOAI2025_ML_Challenge_trainset.csv` (labelled), `IOAI2025_ML_Challenge_testset.csv`
(no label). The hidden test labels are the grading key; **Submit** scores your `submission.csv`
(`id,storey_range`) by accuracy. Runnable baseline below — engineer features to improve.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

train = pd.read_csv('IOAI2025_ML_Challenge_trainset.csv').rename(columns={'Unnamed: 0': 'id'})
test = pd.read_csv('IOAI2025_ML_Challenge_testset.csv').rename(columns={'Unnamed: 0': 'id'})
TARGET = 'storey_range'
test_ids = test['id'].to_numpy()
train.shape, test.shape, train[TARGET].nunique()

## Light feature engineering

Parse `month`→year/mon, `remaining_lease`→months; ordinal-encode categoricals; drop high-cardinality `block`/`street_name`.

In [ ]:
def lease_to_months(s):
    if not isinstance(s, str):
        return np.nan
    import re
    y = re.search(r'(\d+)\s*year', s); m = re.search(r'(\d+)\s*month', s)
    return (int(y.group(1)) if y else 0) * 12 + (int(m.group(1)) if m else 0)

def fe(df):
    d = df.copy()
    dt = pd.to_datetime(d['month'], errors='coerce')
    d['year'] = dt.dt.year
    d['mon'] = dt.dt.month
    d['lease_months'] = d['remaining_lease'].apply(lease_to_months)
    return d.drop(columns=['month', 'remaining_lease', 'block', 'street_name', 'id'])

Xtr_df = fe(train.drop(columns=[TARGET]))
Xte_df = fe(test)
cat_cols = [c for c in Xtr_df.columns if Xtr_df[c].dtype == 'object']
num_cols = [c for c in Xtr_df.columns if c not in cat_cols]

def prep(d, enc=None, fit=False):
    Xc = d[cat_cols].astype('object').where(d[cat_cols].notna(), 'NA')
    if fit:
        enc = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        Xc = enc.fit_transform(Xc)
    else:
        Xc = enc.transform(Xc)
    return np.hstack([d[num_cols].to_numpy(dtype=float), Xc]), enc

X_all, enc = prep(Xtr_df, fit=True)
y_all = train[TARGET].to_numpy()
X_test, _ = prep(Xte_df, enc=enc)
cat_cols, num_cols

## Self-check on a held-out split, then train on all data

In [ ]:
Xtr, Xva, ytr, yva = train_test_split(X_all, y_all, test_size=0.2, random_state=42, stratify=y_all)
clf = HistGradientBoostingClassifier(random_state=42, max_iter=200)
clf.fit(Xtr, ytr)
print(f'Held-out validation accuracy: {accuracy_score(yva, clf.predict(Xva)):.4f}')

In [ ]:
final = HistGradientBoostingClassifier(random_state=42, max_iter=200)
final.fit(X_all, y_all)
preds = final.predict(X_test)
sub = pd.DataFrame({'id': test_ids, 'storey_range': preds})
sub.to_csv('submission.csv', index=False)
print('wrote submission.csv', sub.shape)
sub.head()

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)